# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adityag30/FlyRank-internship-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents one content page (`content_hash_id`) on one reporting day (`report_date`).

**Time window:** March 2026 (`month = '2026-03'`), covering **2026-03-01 to 2026-03-31**.

**Verification:** The dataset contains **9,841,378 rows** and **331,437 unique content pages** for this period.

In [5]:
# This cell is for CODE (numbers, a query, a check).
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")



In [6]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [10]:
query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content_pages,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
"""

unit_df = con.sql(query).df()
unit_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_content_pages,start_date,end_date
0,9841378,331437,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_pageviews`
- `ga4_sessions`
- `ga4_users`
- `ga4_engaged_sessions`
- `ga4_total_engagement_sec`
- `sessions_organic`
- `sessions_direct`
- `sessions_referral`
- `sessions_social`
- `sessions_paid`
- `sessions_ai`
- `scroll_events`

These fields describe search visibility, user engagement, and traffic sources, making them suitable predictive features.

### Label
- No direct label exists in this table. The notebook focuses on defining the data contract, so a future refresh score or ranking target would be created separately.

### Context
- `report_date`
- `month`
- `client_hash_id`
- `content_hash_id`
- `client_has_gsc`
- `client_has_ga4`
- `gsc_data_available`
- `ga4_data_available`

These fields identify the content, client, reporting period, and data availability.

### Excluded
- `ai_chatgpt`
- `ai_perplexity`
- `ai_gemini`
- `ai_copilot`
- `ai_claude`
- `ai_meta`
- `ai_other`

These AI-specific traffic source fields are excluded because they are highly sparse, platform-specific, and not essential for the initial refresh-scoring data contract.

In [11]:
# This cell is for CODE (numbers, a query, a check).
query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content,
    SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS rows_with_gsc,
    SUM(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS rows_with_ga4
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_content,rows_with_gsc,rows_with_ga4
0,9841378,331437,3611061.0,413966.0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

1. The grain is one row per content page per reporting day.
2. The dataset covers the expected time window (March 2026).
3. The key feature fields have acceptable completeness.
4. GSC and GA4 availability flags are present for the selected rows.

In [12]:
# This cell is for CODE (numbers, a query, a check).
#verify grain
query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id || '_' || CAST(report_date AS VARCHAR)) AS unique_content_day_rows
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_content_day_rows
0,9841378,9841378


In [13]:
#verify time window
query = f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(DISTINCT report_date) AS reporting_days
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
"""

con.sql(query).df()

,start_date,end_date,reporting_days
0,2026-03-01,2026-03-31,31


In [14]:
#check missing values
query = f"""
SELECT
    SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS missing_gsc_impressions,
    SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS missing_gsc_clicks,
    SUM(CASE WHEN ga4_pageviews IS NULL THEN 1 ELSE 0 END) AS missing_ga4_pageviews,
    SUM(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END) AS missing_ga4_sessions
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,missing_gsc_impressions,missing_gsc_clicks,missing_ga4_pageviews,missing_ga4_sessions
0,0.0,0.0,3018741.0,3018741.0


In [15]:
#verify data availability flags
query = f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS gsc_available_rows,
    SUM(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS ga4_available_rows
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
"""

con.sql(query).df()

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061.0,413966.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset has several limitations that should be considered before modeling:

- **No ground-truth label:** The table contains performance metrics but does not include a direct target indicating whether a page should be refreshed or whether a refresh was successful.
- **Missing analytics data:** Some rows have `gsc_data_available` or `ga4_data_available` set to false, so certain metrics may be missing for those pages.
- **Limited time window:** This analysis focuses only on March 2026, so it does not capture long-term seasonal trends or historical performance changes.
- **Observational data only:** The dataset records what happened but cannot establish causal relationships between content changes and performance.
- **Client differences:** Performance metrics come from many different clients and websites, so traffic patterns and engagement levels are not directly comparable across all content.
- **Sparse AI traffic fields:** Metrics such as `ai_chatgpt`, `ai_claude`, and `ai_gemini` may contain many zero or missing values because not all sites receive traffic from AI platforms.

In [16]:
# This cell is for CODE (numbers, a query, a check).
query = f"""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN gsc_data_available = FALSE THEN 1 ELSE 0 END) AS rows_without_gsc,
    SUM(CASE WHEN ga4_data_available = FALSE THEN 1 ELSE 0 END) AS rows_without_ga4,

    SUM(CASE WHEN ai_chatgpt > 0 THEN 1 ELSE 0 END) AS rows_with_chatgpt,
    SUM(CASE WHEN ai_gemini > 0 THEN 1 ELSE 0 END) AS rows_with_gemini,
    SUM(CASE WHEN ai_claude > 0 THEN 1 ELSE 0 END) AS rows_with_claude

FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_without_gsc,rows_without_ga4,rows_with_chatgpt,rows_with_gemini,rows_with_claude
0,9841378,6230317.0,6408671.0,3177.0,1611.0,72.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.